### Deze notebook verplaatst HydraRing uitvoer naar een overzichtsbestand waar Jan S. een plot mee kan maken.

In [1]:
from utils.readers import read_design_table
import numpy as np

In [2]:
import zipfile
import re
from pathlib import Path


def read_coordinates(full_path: str) -> tuple[float, float]:
    """
    Extract X and Y RD coordinates from a .txt file nested inside a .zip archive.

    Args:
        full_path: Full path string containing the .zip path and the internal
                   path to the .txt file, e.g.:
                   'C:\\Users\\...\\archive.zip\\internal\\path\\file.txt'

    Returns:
        Tuple of (x_coord, y_coord) as floats.

    Raises:
        FileNotFoundError: If the .zip or internal file cannot be found.
        ValueError: If coordinates cannot be parsed from the file.
    """
    # Split the path into the .zip part and the internal path
    full_path_str = str(full_path)
    zip_end = full_path_str.lower().find('.zip') + len('.zip')
    zip_path = full_path_str[:zip_end]
    internal_path = full_path_str[zip_end:].lstrip('\\/').replace('\\', '/')

    x_coord = None
    y_coord = None

    with zipfile.ZipFile(zip_path, 'r') as zf:
        with zf.open(internal_path) as f:
            for raw_line in f:
                line = raw_line.decode('utf-8', errors='replace')

                if x_coord is None:
                    match = re.search(r'X coord\. section \[m\] RD\s*:\s*([\d.]+)', line)
                    if match:
                        x_coord = float(match.group(1))

                if y_coord is None:
                    match = re.search(r'Y coord\. section \[m\] RD\s*:\s*([\d.]+)', line)
                    if match:
                        y_coord = float(match.group(1))

                if x_coord is not None and y_coord is not None:
                    break  # No need to read further

    if x_coord is None or y_coord is None:
        raise ValueError(
            f"Could not find coordinates in '{internal_path}'. "
            f"Found: X={x_coord}, Y={y_coord}"
        )

    return x_coord, y_coord

In [3]:
type_loc = 'oever' # 'oever', 'as'

from zipfile import ZipFile
from pathlib import PurePosixPath

zip_path = rf"C:\Users\Molendijk\HKV\PR5542.10 - BOI - Verschilanalyse Hydraulische Belastingen - Projectuitvoering - Projectuitvoering\WP02a Beoordelen Oosterschelde\Rekenplan\{type_loc}locaties - concept_20260621\uitvoer\HydraRing_BI2023_KST_Oosterschelde_{type_loc}.zip"

with ZipFile(zip_path, "r") as z:
    design_table_paths = []

    for member in z.namelist():
        path = PurePosixPath(member)

        # Check if the file is inside an 'uitvoer' folder
        if "uitvoer" in path.parts:
            filename = path.name

            if filename.startswith("designTable"):
                design_table_paths.append(str(path))

In [130]:
import pandas as pd
type_parameter = 'WS'
type_som = 'BI2017-totB2017-met'
asked_return_periods = [10, 100, 1000, 3000, 1e4, 3e4, 1e5, 3e5, 1e6, 3e6, 1e7, 3e7, 1e8, 3e8, 1e9, 3e9, 1e10]

columns = [
    'Locatie',
    'X-coördinaat',
    'Y-coördinaat',
    'Terugkeertijd [jaar]',
    'Belastingniveau [m+NAP]/Golfparameter [m]/[s]/Sterkte bekleding [-]'
]

rows = []

for path in design_table_paths:
    locatie_berekening = path.split('/')[1]
    locatie = locatie_berekening.split('_BI2')[0]
    berekening = locatie_berekening.split('_')[-2]

    if berekening != type_som:
        continue
    if path.split('_')[-1] != f"{type_parameter}.txt":
        continue

    return_period, water_level = read_design_table(rf"{zip_path}\{path}")
    requested_return_periods = np.interp(
        np.log(asked_return_periods),
        np.log(return_period),
        water_level
    )

    temp = path.split('/')
    temp[-1] = 'hydraring-input.txt'
    hydraring_input_path = "/".join(temp)
    x, y = read_coordinates(f"{zip_path}\\{hydraring_input_path}")

    for t, v in zip(asked_return_periods, requested_return_periods):
        rows.append({
            'Locatie': locatie,
            'X-coördinaat': x,
            'Y-coördinaat': y,
            'Terugkeertijd [jaar]': t,
            'Belastingniveau [m+NAP]/Golfparameter [m]/[s]/Sterkte bekleding [-]': v,
        })

df = pd.DataFrame(rows, columns=columns)
temp = zip_path.split('\\')
temp = temp[:-1]
save_path = '\\'.join(temp)

df.to_csv(f'{save_path}\\{type_som}_{type_parameter}.csv', index=False, encoding='utf-8-sig')

### Onderstaande itereert over type parameters en sommen

In [4]:
# Voor oeverlocaties is er een mismatch voor de naam van de parameters, deze dict corrigeert dat
correctie_dict = {'go':'HBN', 'hs':'Hs', 'tp':'Tp', 'ts':'Tm01', 'ws':'WS'}

In [7]:
import pandas as pd
type_parameter = 'ws' 
type_som = 'BI2023-totB2023-zon'
asked_return_periods = [10, 100, 1000, 3000, 1e4, 3e4, 1e5, 3e5, 1e6, 3e6, 1e7, 3e7, 1e8]

for type_parameter in ['ws', 'go','hs','tp','ts']:#['ws', 'hs','go','tp','ts']:
    for type_som in ['BI2023-totB2023-zon', 'BI2023-totB2023-met', 'BI2017-totB2017-zon', 'BI2017-totB2017-met']:

        columns = [
            'Locatie',
            'X-coördinaat',
            'Y-coördinaat',
            'Terugkeertijd [jaar]',
            'Belastingniveau [m+NAP]/Golfparameter [m]/[s]/Sterkte bekleding [-]'
        ]

        rows = []

        for path in design_table_paths:
            locatie_berekening = path.split('/')[1]
            locatie = locatie_berekening.split('_BI2')[0]
            berekening = locatie_berekening.split('_')[-2]

            if berekening != type_som:
                continue
            if path.split('_')[-1] != f"{correctie_dict[type_parameter]}.txt":
                continue

            return_period, water_level = read_design_table(rf"{zip_path}\{path}")
            requested_return_periods = np.interp(
                np.log(asked_return_periods),
                np.log(return_period),
                water_level
            )

            temp = path.split('/')
            temp[-1] = 'hydraring-input.txt'
            hydraring_input_path = "/".join(temp)
            x, y = read_coordinates(f"{zip_path}\\{hydraring_input_path}")

            for t, v in zip(asked_return_periods, requested_return_periods):
                rows.append({
                    'Locatie': locatie,
                    'X-coördinaat': x,
                    'Y-coördinaat': y,
                    'Terugkeertijd [jaar]': t,
                    'Belastingniveau [m+NAP]/Golfparameter [m]/[s]/Sterkte bekleding [-]': v,
                })

        df = pd.DataFrame(rows, columns=columns)
        temp = zip_path.split('\\')
        temp = temp[:-1]
        save_path = '\\'.join(temp)
        # print(hydraring_input_path)
        df.to_csv(f'{save_path}\\{type_som}_{correctie_dict[type_parameter]}.csv', index=False, encoding='utf-8-sig')